# Dynamics parity: the pilot's checkpoint and the pilot v2 seed-0 checkpoint on the same split

**Why.** The one-seed B2 probe (`droid100_adjoint_v2_20260930T165409Z`, audit `docs/audits/2026-09-30_b2_probe_seed0_dynamics_gate_audit.md`) failed the dynamics gate on the **validation** split: model RMSE 0.2125 against persistence 0.1886. The pilot's pass (+31 %, RMSE 0.1556 against 0.2262) was measured on the **test** split, with a different training regime (`mask_mode='subset'`, gate in `prediction_mode='full'`; pilot v2 trains `single_choice` and gates `base`). The two numbers are not comparable. This notebook scores both checkpoints on the same windows.

**What it does (no training).** For the pilot checkpoint (`best_dynamics.pt`, hash-checked) and the pilot v2 seed-0 checkpoint (`best.pt`, hash-checked): horizon-mean RMSE against persistence on the **train** and **validation** windows, in both prediction modes (`base` and `full`), each checkpoint in its own input normalisation. It reads the feature cache of the pilot v2 run (no TensorFlow Datasets, no download) and checks the episode split against the pilot's manifest.

**The test split is never built or read.** The B2 design reads test once, at the end, and this diagnostic must not spend it.

**Anchors (checked before any reading is made).** (1) The recomputed v2 validation `base` RMSE must reproduce `artifacts/dynamics_gate.json` of the probe run; (2) the recomputed pilot validation numbers must reproduce the `best_metrics` stored inside the pilot checkpoint. If an anchor fails the notebook reports `ANCHOR_FAILED`, writes the table, and does not offer a reading: a pipeline that cannot reproduce a stored number says nothing about it. The two tolerances (`anchor_atol`, `anchor_atol_v1`) are policy values fixed in the frozen configuration below, not measured. Every row is scored twice, with BF16 autocast (the pipeline of pilot v2's gate; used for the v2 anchor and for the readings) and in FP32 (the pilot's own loop; used for the pilot anchor).

**How to read the result (mechanical, descriptive, not causal).**
- *Pilot checkpoint fails the 2 % margin on validation in `full` mode:* the failure is a property of the validation split and this metric, not of the v2 training run.
- *Pilot checkpoint passes on validation and the v2 checkpoint fails:* the training regime (or the single seed) differs; the v2 checkpoint is worse on the same windows.
- *v2 fails in `base` but passes in `full`:* the gate's prediction mode is what differs.
- Anything else: read the table; no classification is made.

**Limits.** One v2 seed; train and validation only; horizon-mean RMSE in normalised units (comparable between checkpoints only as the relative improvement over persistence on the same windows); it does not explain *why* a difference exists. Nothing here is evidence for or against H2.


In [ ]:
# ================================================================
# 0. Fetch the tested package and pin its commit
# ================================================================
import os
import subprocess
import sys

REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA. The job worker pins the commit here.
REPO_DIR = '/content/para_001'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)


def resolve_ref(ref):
    for candidate in (f'origin/{ref}', ref):
        result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'], capture_output=True, text=True)
        if result.returncode == 0:
            return result.stdout.strip()
    raise RuntimeError(f'Cannot resolve {ref!r} in {REPO_URL}.')


subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolve_ref(REPO_REF)], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

try:
    import adjointrwm
    import adjointrwm.models
    import adjointrwm.training  # noqa: F401
    from adjointrwm.eval import dynamics_gate_row  # noqa: F401
except ImportError as error:
    raise RuntimeError(f'{REPO_REF!r} ({REPO_COMMIT[:12]}) predates adjointrwm.eval.dynamics_parity. Set REPO_REF to a commit that contains it.') from error
print('adjointrwm', adjointrwm.__version__, 'at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '')

In [ ]:
# ================================================================
# 1. Runtime, frozen configuration, run directory
# ================================================================
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

import dataclasses
import datetime
import json
import platform
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from adjointrwm.io import atomic_write_json, atomic_write_text, sha256_file, sha256_json

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DEVICE = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name(0)
if 'L4' not in GPU_NAME:
    raise RuntimeError(f'This diagnostic is scheduled for an L4; this runtime has {GPU_NAME}.')


@dataclass(frozen=True)
class ParityConfig:
    # The two checkpoints (hashes from results/runs/droid100_adjoint_20260929T070629Z/README.md and the probe run's DONE.json).
    v1_run_id: str = 'droid100_adjoint_20260929T070629Z'
    v1_checkpoint_sha256: str = '4b0f07177221ae63467bd0ce7f8ae97545e9fc92a27b3a0731551648709fcc3d'
    v2_run_id: str = 'droid100_adjoint_v2_20260930T165409Z'
    v2_checkpoint_sha256: str = '66c80f464a442da0ba9e3ee424aef692a1be4b76eff860fe1d8640c08555882b'
    v2_seed: int = 0
    # Evaluation: the test split is deliberately not in this list and is never built.
    splits: tuple = ('train', 'validation')
    modes: tuple = ('base', 'full')
    amp_modes: tuple = (True, False)   # True: BF16 autocast as in pilot v2's gate; False: FP32 as in the pilot's own evaluation loop
    frame_stride: int = 2
    context_len: int = 8
    horizon: int = 4
    window_stride: int = 2
    eval_batch_size: int = 128
    margin: float = 0.02            # the gate's required relative improvement over persistence
    # Policy values, not measurements: how closely a recomputed number must reproduce a stored one.
    anchor_atol: float = 1.0e-3      # v2 validation base RMSE against the probe run's dynamics_gate.json (same code path)
    anchor_atol_v1: float = 5.0e-3   # pilot validation numbers against best_metrics inside its checkpoint (evaluated by the pilot's own loop)
    normaliser_atol: float = 1.0e-3  # above this the two checkpoints' input units differ visibly; reported, not fatal


CFG = ParityConfig()
assert 'test' not in CFG.splits
CONFIG_DICT = asdict(CFG)
CONFIG_HASH = sha256_json(CONFIG_DICT)

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production')
SESSION_ID = datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
RUN_ID = 'dynamics_parity_' + SESSION_ID
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_ID
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} already exists; run directories are immutable.')
PATHS = {name: RUN_DIR / name for name in ('config', 'artifacts', 'reports', 'tests')}
LOCAL = Path('/content/adjoint_rwm_work') / RUN_ID
for path in [LOCAL, *PATHS.values()]:
    path.mkdir(parents=True, exist_ok=True)

atomic_write_json(PATHS['config'] / 'run_config.json', {
    'run_id': RUN_ID, 'config': CONFIG_DICT, 'config_sha256': CONFIG_HASH, 'repo_commit': REPO_COMMIT, 'repo_dirty': REPO_DIRTY,
    'gpu': GPU_NAME, 'notebook': 'notebooks/02-diagnostics/dynamics_parity.ipynb', 'test_split_read': False,
    'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__,
})
print('Run:', RUN_ID, '| GPU:', GPU_NAME, '| config', CONFIG_HASH[:12])

In [ ]:
# ================================================================
# 2. Feature cache of the pilot v2 run -> train and validation windows (no test split, no download)
# ================================================================
from adjointrwm.data import WindowDataset, WindowSpec, assignment_from_manifest, compare_splits, episode_split, fit_normaliser, restore_cache, split_sets

V1_DIR = DRIVE_ROOT / 'runs' / CFG.v1_run_id
V2_DIR = DRIVE_ROOT / 'runs' / CFG.v2_run_id

cache_manifest = json.loads((V2_DIR / 'cache' / 'cache_manifest.json').read_text())
records = restore_cache(cache_manifest, LOCAL / 'cache')
print(f'Restored {len(records)} cached episodes (hashes verified).')

assignment = episode_split([r['episode_id'] for r in records])
for record in records:
    record['split'] = assignment[record['episode_id']]
SPLIT_PARITY = compare_splits(assignment, assignment_from_manifest(json.loads((V1_DIR / 'config' / 'data_manifest.json').read_text())))
if not SPLIT_PARITY['matches']:
    raise RuntimeError(f'The episode split differs from the pilot: {SPLIT_PARITY}')
sets = split_sets(assignment)
assert sets['train'].isdisjoint(sets['validation']) and sets['train'].isdisjoint(sets['test']) and sets['validation'].isdisjoint(sets['test'])

SPEC = WindowSpec(CFG.context_len, CFG.horizon, CFG.window_stride)
BY_SPLIT = {s: [r for r in records if r['split'] == s] for s in CFG.splits}
assert set(BY_SPLIT) == {'train', 'validation'}, 'the test split must not be built'


def load_arrays(record):
    with np.load(record['cached_path']) as episode:
        return {k: episode[k] for k in ('states', 'actions')}


train_arrays = [load_arrays(r) for r in BY_SPLIT['train']]
NORM_V2 = fit_normaliser([a['states'] for a in train_arrays], [a['actions'] for a in train_arrays])
print('Episodes per split:', {s: len(v) for s, v in BY_SPLIT.items()}, '| split parity with the pilot:', SPLIT_PARITY['matches'])

In [ ]:
# ================================================================
# 3. The two checkpoints: hash check, payload, each one's own input normaliser, windows
# ================================================================
from adjointrwm.eval import anchor_check, compare_normalisers, dynamics_gate_row, stack_windows
from adjointrwm.models import AdjointRecursiveWorldModel, AdjointRWMConfig
from adjointrwm.training import predict_dataset

CHECKPOINTS = {
    'v1': (V1_DIR / 'checkpoints' / 'best_dynamics.pt', CFG.v1_checkpoint_sha256, 'subset'),
    'v2': (V2_DIR / 'jobs' / f'seed_{CFG.v2_seed}' / 'dynamics' / 'best.pt', CFG.v2_checkpoint_sha256, 'single_choice'),
}
PAYLOADS, CHECKPOINT_FACTS = {}, {}
for label, (path, expected, mask_mode) in CHECKPOINTS.items():
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f'{label}: {path} has SHA-256 {actual}, expected {expected}. Not the checkpoint this diagnostic was written for.')
    payload = torch.load(path, map_location='cpu', weights_only=False)   # our own files, hash-checked just above
    PAYLOADS[label] = payload
    CHECKPOINT_FACTS[label] = {'path': str(path), 'sha256': actual, 'global_step': payload.get('global_step'), 'mask_mode_trained_with': mask_mode,
                               'payload_keys': sorted(payload), 'has_stored_normalisation': payload.get('normalisation') is not None}
    print(label, CHECKPOINT_FACTS[label])

NORM_V1 = PAYLOADS['v1'].get('normalisation')
NORMALISER_COMPARISON = compare_normalisers(NORM_V1, NORM_V2) if NORM_V1 is not None else None
print('normaliser v1 against v2 (refit from the same train split):', NORMALISER_COMPARISON)
NORMS = {'v1': NORM_V1 if NORM_V1 is not None else NORM_V2, 'v2': NORM_V2}   # each checkpoint in its own native input units
DATASETS = {label: {s: WindowDataset(BY_SPLIT[s], SPEC, NORMS[label], visual_layout='flat') for s in CFG.splits} for label in NORMS}
print('Windows:', {s: len(d) for s, d in DATASETS['v2'].items()})

sample = DATASETS['v2']['train'][0]
STATE_DIM, ACTION_DIM, VISUAL_DIM = sample['context_state'].shape[-1], sample['context_action'].shape[-1], sample['context_visual'].shape[-1]
V1_CONFIG = PAYLOADS['v1'].get('config', {})
MODEL_FIELDS = dict(context_len=CFG.context_len, horizon=CFG.horizon, d_model=512, transformer_layers=6, transformer_heads=8, transformer_ff=2048, dropout=0.10,
                    num_refinement_candidates=4, candidate_costs=(1.0, 1.0, 1.5, 2.0), rate_beta=0.002)
CONFIG_MISMATCH_V1 = {k: (V1_CONFIG[k], v) for k, v in MODEL_FIELDS.items() if k in V1_CONFIG and list(np.ravel(V1_CONFIG[k])) != list(np.ravel(v))}
print('model fields that differ from the pilot checkpoint\'s stored config:', CONFIG_MISMATCH_V1 or 'none')


def build_model(mask_mode):
    return AdjointRecursiveWorldModel(STATE_DIM, ACTION_DIM, VISUAL_DIM, AdjointRWMConfig(mask_mode=mask_mode, prediction_mode='base', **MODEL_FIELDS))

In [ ]:
# ================================================================
# 4. Score both checkpoints against persistence: train and validation, base and full
# ================================================================
ROWS, LOAD_ERRORS = [], {}
for label, (_, _, mask_mode) in CHECKPOINTS.items():
    model = build_model(mask_mode).to(DEVICE)
    try:
        model.load_state_dict(PAYLOADS[label]['model_state_dict'], strict=True)
    except RuntimeError as error:   # recorded, not hidden: a checkpoint that does not load into this model cannot be scored
        LOAD_ERRORS[label] = str(error)[:2000]
        print(label, 'did NOT load:', LOAD_ERRORS[label][:300])
        continue
    model.eval()
    for mode in CFG.modes:
        model.config = dataclasses.replace(model.config, prediction_mode=mode)
        for split in CFG.splits:
            dataset = DATASETS[label][split]
            context = stack_windows(dataset, keys=['context_state'])['context_state']
            for amp in CFG.amp_modes:
                pred = predict_dataset(model, dataset, CFG.eval_batch_size, DEVICE, amp=amp)
                ROWS.append({'checkpoint': label, 'prediction_mode': mode, 'split': split, 'amp': amp,
                             **dynamics_gate_row(pred['state_mean'], pred['target_state'], context, CFG.horizon, CFG.margin)})
    del model
    torch.cuda.empty_cache()

TABLE = pd.DataFrame(ROWS)
view = ['checkpoint', 'prediction_mode', 'split', 'amp', 'windows', 'model_rmse', 'persistence_rmse', 'relative_improvement', 'passed']
print(TABLE[view].to_string(index=False))
print()
print(TABLE.assign(improvement_by_horizon=TABLE['improvement_by_horizon'].map(lambda xs: [round(x, 3) for x in xs]))[['checkpoint', 'prediction_mode', 'split', 'amp', 'improvement_by_horizon']].to_string(index=False))
atomic_write_json(PATHS['artifacts'] / 'dynamics_parity.json', ROWS)
TABLE[view].to_csv(PATHS['artifacts'] / 'dynamics_parity.csv', index=False)

In [ ]:
# ================================================================
# 5. Anchors, mechanical readings, report
# ================================================================
def cell(label, mode, split, amp=True):
    found = [r for r in ROWS if (r['checkpoint'], r['prediction_mode'], r['split'], r['amp']) == (label, mode, split, amp)]
    return found[0] if found else None


def number(mapping, name):
    """A scalar stored under ``name`` or the mean of its ``name_by_horizon`` list, searching nested dictionaries."""
    if isinstance(mapping, dict):
        if name in mapping and np.ndim(mapping[name]) == 0:
            return float(mapping[name])
        if f'{name}_by_horizon' in mapping:
            return float(np.mean(np.asarray(mapping[f'{name}_by_horizon'], dtype=np.float64)))
        for value in mapping.values():
            found = number(value, name) if isinstance(value, dict) else None
            if found is not None:
                return found
    return None


ANCHORS = {}
stored_v2 = json.loads((V2_DIR / 'artifacts' / 'dynamics_gate.json').read_text())[str(CFG.v2_seed)]
v2_base_val = cell('v2', 'base', 'validation')
if v2_base_val is not None:
    ANCHORS['v2_validation_base_rmse_vs_dynamics_gate_json'] = anchor_check(v2_base_val['model_rmse'], stored_v2['model_rmse'], CFG.anchor_atol)
    ANCHORS['v2_validation_persistence_rmse_vs_dynamics_gate_json'] = anchor_check(v2_base_val['persistence_rmse'], stored_v2['persistence_rmse'], CFG.anchor_atol)
best_v1 = PAYLOADS['v1'].get('best_metrics') or {}
for name, (mode, field) in {'full_rmse': ('full', 'model_rmse'), 'base_rmse': ('base', 'model_rmse'), 'persistence_rmse': ('full', 'persistence_rmse')}.items():
    stored, row = number(best_v1, name), cell('v1', mode, 'validation', amp=False)   # the pilot's own loop ran in FP32
    if stored is not None and row is not None:
        ANCHORS[f'v1_validation_{name}_vs_stored_best_metrics'] = anchor_check(row[field], stored, CFG.anchor_atol_v1)
ANCHORS_OK = bool(ANCHORS) and all(a['within'] for a in ANCHORS.values())
print(json.dumps(ANCHORS, indent=1))
print('pilot best_metrics (validation, as stored in its checkpoint):', json.dumps(best_v1, default=str)[:600])

if LOAD_ERRORS or not ANCHORS_OK or any(cell(*k) is None for k in [('v1', 'full', 'validation'), ('v2', 'base', 'validation'), ('v2', 'full', 'validation')]):
    status, READINGS = ('ANCHOR_FAILED' if ANCHORS and not ANCHORS_OK else 'INCOMPLETE'), []
else:
    status = 'OK'
    v1f, v2b, v2f = cell('v1', 'full', 'validation'), cell('v2', 'base', 'validation'), cell('v2', 'full', 'validation')   # readings: the BF16 pipeline of pilot v2's gate
    READINGS = []
    if not v1f['passed']:
        READINGS.append('The pilot checkpoint also fails the margin on validation in full mode: the failure is a property of the validation split and this metric, not of the v2 training run.')
    elif not v2b['passed']:
        READINGS.append('The pilot checkpoint passes on validation in full mode and the v2 checkpoint fails in base mode: the v2 checkpoint is worse on the same windows '
                        '(training regime or the single seed).')
    if not v2b['passed'] and v2f['passed']:
        READINGS.append('The v2 checkpoint fails in base mode but passes in full mode: the prediction mode of the gate is what differs.')
    if not READINGS:
        READINGS.append('No mechanical reading applies (see the table).')
for line in READINGS:
    print('-', line)

REPORT = {
    'run_id': RUN_ID, 'status': status, 'config_hash': CONFIG_HASH, 'repo_commit': REPO_COMMIT, 'test_split_read': False,
    'checkpoints': CHECKPOINT_FACTS, 'load_errors': LOAD_ERRORS, 'model_fields_differing_from_pilot_config': CONFIG_MISMATCH_V1,
    'normaliser_comparison': NORMALISER_COMPARISON, 'split_parity_with_pilot': SPLIT_PARITY, 'anchors': ANCHORS, 'anchors_ok': ANCHORS_OK,
    'rows': ROWS, 'readings': READINGS,
    'claim_boundary': ('Diagnostic of dynamics against persistence on train and validation windows for two checkpoints (one pilot v2 seed). '
                       'Not evidence for or against H2; the test split was not read.'),
}
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)
lines = [f'# Dynamics parity `{RUN_ID}`', '', f'Status: **{status}**. Anchors within tolerance: {ANCHORS_OK}.', '',
         '| Checkpoint | Mode | Split | BF16 | Windows | Model RMSE | Persistence RMSE | Relative improvement | Passes 2 % |', '|---|---|---|---|---:|---:|---:|---:|---|']
for r in ROWS:
    lines.append(f"| {r['checkpoint']} | {r['prediction_mode']} | {r['split']} | {r['amp']} | {r['windows']} | {r['model_rmse']:.4f} | {r['persistence_rmse']:.4f} | "
                 f"{r['relative_improvement']:+.3f} | {r['passed']} |")
lines += ['', *(f'- {line}' for line in READINGS), '', REPORT['claim_boundary']]
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join(lines) + '\n')
if status == 'OK':
    atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print('\n'.join(lines))
print('\nRun directory:', RUN_DIR)